## Handle 7/9, 77/99, 777/999, 7777/9999, 77777/99999 code in columns.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("nhanes_64_col.csv")

# Refused / don't-know codes for each column.
# The code depends on how many digits the real answers use:
#   answers 0-9      -> 7 = refused, 9 = don't know
#   answers 0-99     -> 77 / 99
#   answers 0-999    -> 777 / 999
#   answers 0-9999   -> 7777 / 9999
# Columns NOT listed here (age, race, household size, sleep hours, ...) have no codes:
# a 7 or 77 in those columns is a real value.
missing_codes = {
    # --- Demographics ---
    "Served active duty in US Armed Forces":    [7, 9],
    "Education level - Adults 20+":             [7, 9],
    "Marital status":                           [77, 99],
    "Length of time in US":                     [77, 99],

    # --- Physical activity (real answers go up to hundreds of minutes) ---
    "Frequency of moderate LTPA":               [7777, 9999],
    "Minutes moderate LTPA":                    [7777, 9999],
    "Frequency of vigorous LTPA":               [7777, 9999],
    "Minutes vigorous LTPA":                    [7777, 9999],
    "Minutes sedentary activity":               [7777, 9999],

    # --- Diet behavior (1 = yes, 2 = no) ---
    "Main meal planner/preparer":               [7, 9],
    "Shared meal planning/preparing duty":      [7, 9],
    "Main food shopper":                        [7, 9],
    "Shared food shopping duty":                [7, 9],

    # --- Smoking ---
    "Smoked at least 100 cigarettes in life":   [7, 9],
    "Do you now smoke cigarettes?":             [7, 9],
    "# days smoked cigs during past 30 days":   [77, 99],
    "Avg # cigarettes/day during past 30 days": [777, 999],
    "Cigarette Menthol indicator":              [9],       # check codebook: assumed 9 = don't know

    # --- Alcohol (frequency categories run 0-10, so 7 and 9 are REAL answers) ---
    "Ever had a drink of any kind of alcohol":  [7, 9],
    "Past 12 mos how often drink alc bev":      [77, 99],
    "Avg # alcoholic drinks/day/past 12 mos":   [777, 999],
    "# days have 4/5 drinks/past 12 mos":       [77, 99],
    "# times 4/5 drinks in 2hrs/past 12 mos":   [77, 99],
    "# times 8+ drinks in 1 day/past 12 mos":   [77, 99],
    "Ever have 4/5 or more drinks every day":   [7, 9],
    "# times 4/5 drinks on occasion/past mo":   [777, 999],

    # --- PHQ-9 depression items (answers 0-3) ---
    "Have little interest in doing things":     [7, 9],
    "Feeling down, depressed, or hopeless":     [7, 9],
    "Trouble sleeping or sleeping too much":    [7, 9],
    "Feeling tired or having little energy":    [7, 9],
    "Poor appetite or overeating":              [7, 9],
    "Feeling bad about yourself":               [7, 9],
    "Trouble concentrating on things":          [7, 9],
    "Moving or speaking slowly or too fast":    [7, 9],
    "Thought you would be better off dead":     [7, 9],
    "Difficulty these problems have caused":    [7, 9],

    # --- Weight history (77 inches is a real height, so only 7777/9999) ---
    "Current self-reported height (inches)":    [7777, 9999],
    "Current self-reported weight (pounds)":    [7777, 9999],
    "Self-reported weight - 1 yr ago (pounds)": [7777, 9999],
}

# Sleep clock times are text ("22:30"), with "77777" / "99999" as codes
text_codes = {
    "Usual sleep time on weekdays or workdays": ["77777", "99999"],
    "Usual wake time on weekdays or workdays":  ["77777", "99999"],
    "Usual sleep time on weekends":             ["77777", "99999"],
    "Usual wake time on weekends":              ["77777", "99999"],
}

# Replace the codes with NaN, keeping a record of what changed
report = []
for col, codes in {**missing_codes, **text_codes}.items():
    if col not in df.columns:
        continue                                  # skip columns you've already dropped
    for code in codes:
        n = (df[col] == code).sum()
        if n > 0:
            report.append({"column": col, "code": code, "n_set_to_missing": n})
    df[col] = df[col].replace(codes, np.nan)

report = pd.DataFrame(report)
print(f"Replaced {report['n_set_to_missing'].sum()} coded values with NaN "
      f"across {report['column'].nunique()} columns")
report

Replaced 442 coded values with NaN across 38 columns


,column,code,n_set_to_missing
0,Served active duty in US Armed Forces,7,1
1,Education level - Adults 20+,9,1
2,Marital status,77,2
3,Marital status,99,2
4,Length of time in US,77,4
5,Length of time in US,99,16
6,Frequency of moderate LTPA,7777,2
7,Frequency of moderate LTPA,9999,15
8,Minutes moderate LTPA,9999,10
9,Frequency of vigorous LTPA,7777,1


In [2]:
print(df.shape)

(3583, 64)


## Drop the rows that have missing current weight or weight 1 year ago, so their % weight change can’t be calculated.

In [3]:
wt_now, wt_before = "Current self-reported weight (pounds)", "Self-reported weight - 1 yr ago (pounds)"
df = df.dropna(subset=[wt_now, wt_before])
print(df.shape)

(3480, 64)


## Fill the skip-pattern zeros (activity, alcohol) and turn raw answers into usable variables: activity minutes/week, Patient Health Questionnaire 9 items (PHQ-9) total score for depression, drinks per week, smoking status, and average sleep hours.

In [4]:
def status(s):
    """Label each value as 'blank' or 'answered' for the crosstabs."""
    return s.notna().map({True: "answered", False: "blank"})

# ACTIVITY: is unit/minutes blank exactly when frequency = 0?
for kind in ["moderate", "vigorous"]:
    freq = df[f"Frequency of {kind} LTPA"]
    unit = df[f"{kind.capitalize()} LTPA unit (day/week/month/year)"]
    mins = df[f"Minutes {kind} LTPA"]

    freq_group = pd.Series("missing", index=df.index)
    freq_group[freq == 0] = "0 sessions"
    freq_group[freq > 0] = "1+ sessions"

    table = pd.crosstab(
        freq_group.rename(f"Frequency of {kind} activity"),
        [status(unit).rename("Unit"), status(mins).rename("Minutes")],
        margins=True, margins_name="Total",
    )
    print(f"\n=== {kind.upper()} ACTIVITY ===")
    display(table)
    n_zero = (freq == 0).sum()
    print(f"{n_zero} people with 0 {kind} sessions -> unit and minutes skipped -> set to 0 minutes")
    print(f"% missing in minutes before: {mins.isna().mean():.1%}")

# ALCOHOL: never-drinkers skip frequency; 0 drinks in past year skip amount
ever_drank = df["Ever had a drink of any kind of alcohol"]          # 1 = yes, 2 = no
how_often  = df["Past 12 mos how often drink alc bev"]              # 0 = never in past year
per_day    = df["Avg # alcoholic drinks/day/past 12 mos"]

ever_label = ever_drank.map({1: "Yes", 2: "No (never drank)"}).fillna("missing")
print("\n=== ALCOHOL: Ever drank vs. past-year frequency ===")
display(pd.crosstab(ever_label.rename("Ever had a drink"),
                    status(how_often).rename("Past-12-month frequency"),
                    margins=True, margins_name="Total"))

often_label = pd.Series("missing", index=df.index)
often_label[how_often == 0] = "0 = never in past year"
often_label[how_often > 0] = "1-10 = drank in past year"
print("\n=== ALCOHOL: Past-year frequency vs. drinks per day ===")
display(pd.crosstab(often_label.rename("Past-12-month frequency"),
                    status(per_day).rename("Drinks per day"),
                    margins=True, margins_name="Total"))

print(f"Never drank: {(ever_drank == 2).sum()} | "
      f"Drank before but not in past year: {(how_often == 0).sum()} -> both set to 0 drinks/week")

# SMOKING: "Do you now smoke" only asked if smoked 100+ cigarettes
smoked_100 = df["Smoked at least 100 cigarettes in life"]   # 1 = yes, 2 = no
smoke_now  = df["Do you now smoke cigarettes?"]             # 1 = every day, 2 = some days, 3 = not at all

print("\n=== SMOKING ===")
display(pd.crosstab(
    smoked_100.map({1: "Yes", 2: "No"}).fillna("missing").rename("Smoked 100+ cigarettes"),
    smoke_now.map({1: "Every day", 2: "Some days", 3: "Not at all"}).fillna("blank")
             .rename("Do you now smoke?"),
    margins=True, margins_name="Total"))
print(f"{(smoked_100 == 2).sum()} people never smoked 100 cigarettes -> question skipped -> 'Never'")


=== MODERATE ACTIVITY ===


Unit                           answered       blank Total
Minutes                        answered blank blank      
Frequency of moderate activity                           
0 sessions                            0     0   612   612
1+ sessions                        2846     9     0  2855
missing                               0     0    13    13
Total                              2846     9   625  3480

612 people with 0 moderate sessions -> unit and minutes skipped -> set to 0 minutes
% missing in minutes before: 18.2%

=== VIGOROUS ACTIVITY ===


Unit                           answered       blank Total
Minutes                        answered blank blank      
Frequency of vigorous activity                           
0 sessions                            0     0  1772  1772
1+ sessions                        1689     3     0  1692
missing                               0     0    16    16
Total                              1689     3  1788  3480

1772 people with 0 vigorous sessions -> unit and minutes skipped -> set to 0 minutes
% missing in minutes before: 51.5%

=== ALCOHOL: Ever drank vs. past-year frequency ===


...


=== ALCOHOL: Past-year frequency vs. drinks per day ===


Drinks per day,answered,blank,Total
Past-12-month frequency,,,
0 = never in past year,0,371,371
1-10 = drank in past year,1890,15,1905
missing,1,1203,1204
Total,1891,1589,3480


Never drank: 191 | Drank before but not in past year: 371 -> both set to 0 drinks/week

=== SMOKING ===


Do you now smoke?,Every day,Not at all,Some days,blank,Total
Smoked 100+ cigarettes,,,,,
No,0,0,0,2170,2170
Yes,280,929,96,0,1305
missing,0,0,0,5,5
Total,280,929,96,2175,3480


2170 people never smoked 100 cigarettes -> question skipped -> 'Never'


In [5]:
print(df.shape)

(3480, 64)


## Change Frequency of moderate LTPA, Moderate LTPA unit (day/week/month/year), Minutes moderate LTPA to continuous values, by adding a new columns and drop these 3 columns.

In [6]:
import numpy as np

freq_col = "Frequency of moderate LTPA"
unit_col = "Moderate LTPA unit (day/week/month/year)"
min_col  = "Minutes moderate LTPA"

# 1. Convert the frequency to sessions per week using the unit
to_per_week = {"D": 7, "W": 1, "M": 7 / 30.4375, "Y": 1 / 52.1786}
sessions_per_week = df[freq_col] * df[unit_col].map(to_per_week)

# 2. Skip pattern: frequency = 0 -> unit and minutes were not asked -> 0
sessions_per_week = sessions_per_week.where(df[freq_col] != 0, 0)
minutes_per_session = df[min_col].where(df[freq_col] != 0, 0)

# 3. Cap implausible values
sessions_per_week   = sessions_per_week.clip(upper=28)      # max 4 sessions per day
minutes_per_session = minutes_per_session.clip(upper=180)   # max 3 hours per session

# 4. New continuous column: moderate minutes per week (capped at 3 hours/day)
df["mod_min_wk"] = (sessions_per_week * minutes_per_session).clip(upper=1260)

# 5. Check before dropping
print(df["mod_min_wk"].describe().round(1))
print(f"% missing: {df['mod_min_wk'].isna().mean():.1%}")
print(f"People with 0 minutes: {(df['mod_min_wk'] == 0).sum()}")

# 6. Drop the 3 original columns
df = df.drop(columns=[freq_col, unit_col, min_col])
print(df.shape)

count    3458.0
mean      189.4
std       241.3
min         0.0
25%        30.0
50%       120.0
75%       240.0
max      1260.0
Name: mod_min_wk, dtype: float64
% missing: 0.6%
People with 0 minutes: 612
(3480, 62)


## Change Frequency of vigorous LTPA, Vigorous LTPA unit (day/week/month/year), Minutes vigorous LTPA to continuous values, by adding a new columns and drop these 3 columns.

In [7]:
import numpy as np

freq_col = "Frequency of vigorous LTPA"
unit_col = "Vigorous LTPA unit (day/week/month/year)"
min_col  = "Minutes vigorous LTPA"

# 1. Convert the frequency to sessions per week using the unit
to_per_week = {"D": 7, "W": 1, "M": 7 / 30.4375, "Y": 1 / 52.1786}
sessions_per_week = df[freq_col] * df[unit_col].map(to_per_week)

# 2. Skip pattern: frequency = 0 -> unit and minutes were not asked -> 0
sessions_per_week = sessions_per_week.where(df[freq_col] != 0, 0)
minutes_per_session = df[min_col].where(df[freq_col] != 0, 0)

# 3. Cap implausible values
sessions_per_week   = sessions_per_week.clip(upper=28)      # max 4 sessions per day
minutes_per_session = minutes_per_session.clip(upper=180)   # max 3 hours per session

# 4. New continuous column: vigorous minutes per week (capped at 3 hours/day)
df["vig_min_wk"] = (sessions_per_week * minutes_per_session).clip(upper=1260)

# 5. Check before dropping
print(df["vig_min_wk"].describe().round(1))
print(f"% missing: {df['vig_min_wk'].isna().mean():.1%}")
print(f"People with 0 minutes: {(df['vig_min_wk'] == 0).sum()}")

# 6. Drop the 3 original columns
df = df.drop(columns=[freq_col, unit_col, min_col])
print(df.shape)

count    3461.0
mean       62.5
std       135.7
min         0.0
25%         0.0
50%         0.0
75%        60.0
max      1260.0
Name: vig_min_wk, dtype: float64
% missing: 0.5%
People with 0 minutes: 1772
(3480, 60)


## Combine both into total activity


In [8]:
df["mvpa_min_wk"] = df["mod_min_wk"] + 2 * df["vig_min_wk"]
df["meets_pa_guideline"] = (df["mvpa_min_wk"] >= 150).astype(float).where(df["mvpa_min_wk"].notna())

MVPA stands for moderate-to-vigorous physical activity. The formula counts each vigorous minute as 2 moderate minutes because vigorous activity is roughly twice as intense.

In [9]:
print(df.shape)

(3480, 62)


## Convert minutes of sedentary activity per day into hours per day, then drops the original column.

In [10]:
sed_col = "Minutes sedentary activity"   # PAD680: minutes sitting on a typical day

# 1. Minutes per day -> hours per day
df["sedentary_hr_day"] = df[sed_col] / 60

# 2. Check before dropping
print(df["sedentary_hr_day"].describe().round(1))
print(f"% missing: {df['sedentary_hr_day'].isna().mean():.1%}")
print(f"Over 18 hours/day: {(df['sedentary_hr_day'] > 18).sum()}")

# 3. Drop the original column
df = df.drop(columns=[sed_col])
print(df.shape)

count    3463.0
mean        6.2
std         3.5
min         0.0
25%         4.0
50%         6.0
75%         8.0
max        22.0
Name: sedentary_hr_day, dtype: float64
% missing: 0.5%
Over 18 hours/day: 9
(3480, 62)


In [11]:
df["sedentary_hr_day"] = df["sedentary_hr_day"].clip(upper=18)

In [12]:
print(df.shape)

(3480, 62)


## Change Have little interest in doing things, Feeling down, depressed, or hopeless, Trouble sleeping or sleeping too much, Feeling tired or having little energy, Poor appetite or overeating, Feeling bad about yourself, Trouble concentrating on things, Moving or speaking slowly or too fast, Thought you would be better off dead into phq9_score, depression_phq10 and also droping the original columns.

In [13]:
import numpy as np
import pandas as pd

phq_items = [
    "Have little interest in doing things",
    "Feeling down, depressed, or hopeless",
    "Trouble sleeping or sleeping too much",
    "Feeling tired or having little energy",
    "Poor appetite or overeating",
    "Feeling bad about yourself",
    "Trouble concentrating on things",
    "Moving or speaking slowly or too fast",
    "Thought you would be better off dead",
]

# 1. How many of the 9 items did each person answer?
print("Items answered per person:")
print(df[phq_items].notna().sum(axis=1).value_counts().sort_index())

# 2. PHQ-9 total score (0-27): only scored when all 9 items are answered
df["phq9_score"] = df[phq_items].sum(axis=1, min_count=9)

# 3. Depression flag: PHQ-9 >= 10 = moderate or worse depressive symptoms (1/0)
df["depression_phq10"] = (df["phq9_score"] >= 10).astype(float).where(df["phq9_score"].notna())

# 4. Check before dropping
print("\nPHQ-9 score:")
print(df["phq9_score"].describe().round(1))
print(f"% missing: {df['phq9_score'].isna().mean():.1%}")
print("\nDepression (PHQ-9 >= 10):")
print(df["depression_phq10"].value_counts(dropna=False))

# 5. Drop the 9 original items
df = df.drop(columns=phq_items)
print(df.shape)

Items answered per person:
0    1000
2       1
5       1
7       3
8      10
9    2465
Name: count, dtype: int64

PHQ-9 score:
count    2465.0
mean        4.2
std         4.7
min         0.0
25%         1.0
50%         3.0
75%         6.0
max        25.0
Name: phq9_score, dtype: float64
% missing: 29.2%

Depression (PHQ-9 >= 10):
depression_phq10
0.0    2142
NaN    1015
1.0     323
Name: count, dtype: int64
(3480, 55)


In [14]:
df["phq9_severity"] = pd.cut(
    df["phq9_score"],
    bins=[-1, 4, 9, 14, 19, 27],
    labels=["Minimal (0-4)", "Mild (5-9)", "Moderate (10-14)",
            "Moderately severe (15-19)", "Severe (20-27)"],
)
print(df["phq9_severity"].value_counts().sort_index())

phq9_severity
Minimal (0-4)                1587
Mild (5-9)                    555
Moderate (10-14)              214
Moderately severe (15-19)      81
Severe (20-27)                 28
Name: count, dtype: int64


In [15]:
print(df.shape)

(3480, 56)


In [16]:
df = df.drop(columns=["Difficulty these problems have caused"], errors="ignore")

In [17]:
print(df.shape)

(3480, 55)


## Change Ever had a drink of any kind of alcohol, Past 12 mos how often drink alc bev, Avg number alcoholic drinks/day/past 12 mos into drinks_per_week, alcohol_status and drop the original columns.

In [18]:
import numpy as np
import pandas as pd

ever_drank = "Ever had a drink of any kind of alcohol"     # 1 = yes, 2 = no
how_often  = "Past 12 mos how often drink alc bev"         # 0-10 frequency code
per_day    = "Avg # alcoholic drinks/day/past 12 mos"      # drinks on a drinking day

# 1. Turn the frequency code into drinking DAYS per week
#    (higher code = LESS drinking, so it can't be used as a number directly)
days_per_week = {
    0: 0,                 # never in the last year
    1: 7,                 # every day
    2: 5.5,               # nearly every day (5-6 days/week)
    3: 3.5,               # 3-4 times a week
    4: 2,                 # 2 times a week
    5: 1,                 # once a week
    6: 2.5 / 4.345,       # 2-3 times a month
    7: 1 / 4.345,         # once a month
    8: 9 / 52.18,         # 7-11 times in the last year
    9: 4.5 / 52.18,       # 3-6 times in the last year
    10: 1.5 / 52.18,      # 1-2 times in the last year
}
drink_days = df[how_often].map(days_per_week)

# 2. Skip patterns
drink_days = drink_days.where(df[ever_drank] != 2, 0)     # never drank -> frequency skipped -> 0 days
drinks = df[per_day].where(drink_days != 0, 0)            # 0 drinking days -> amount skipped -> 0 drinks

# 3. Drinks per week = drinking days per week x drinks per drinking day
df["drinks_per_week"] = (drink_days * drinks).round(2)
cap = df["drinks_per_week"].quantile(0.99)                # limit a few extreme answers
df["drinks_per_week"] = df["drinks_per_week"].clip(upper=cap)

# 4. Drinking status
df["alcohol_status"] = np.select(
    [df[ever_drank] == 2,          # never had a drink
     drink_days == 0,              # drank before, but not in the past 12 months
     drink_days > 0],              # drank in the past 12 months
    ["Never drinker", "Former/past-year abstainer", "Current drinker"],
    default=None,                  # unknown (e.g., not examined)
)

# 5. Check before dropping
print(f"drinks_per_week capped at the 99th percentile: {cap:.1f}")
print(df["drinks_per_week"].describe().round(1))
print(f"% missing: {df['drinks_per_week'].isna().mean():.1%}")
print("\nAlcohol status:")
print(df["alcohol_status"].value_counts(dropna=False))

# 6. Drop the 3 original columns
df = df.drop(columns=[ever_drank, how_often, per_day])
print(df.shape)

drinks_per_week capped at the 99th percentile: 36.7
count    2452.0
mean        3.2
std         6.3
min         0.0
25%         0.0
50%         0.5
75%         3.5
max        36.7
Name: drinks_per_week, dtype: float64
% missing: 29.5%

Alcohol status:
alcohol_status
Current drinker               1905
None                          1013
Former/past-year abstainer     371
Never drinker                  191
Name: count, dtype: int64
(3480, 54)


Drop four extra binge-drinking columns (“# days have 4/5 drinks”, “4/5 drinks in 2hrs”, “8+ drinks”, “4/5 drinks on occasion”) and “Ever have 4/5 or more drinks every day”.

In [19]:
binge_cols = [
    "# days have 4/5 drinks/past 12 mos", "# times 4/5 drinks in 2hrs/past 12 mos",
    "# times 8+ drinks in 1 day/past 12 mos", "# times 4/5 drinks on occasion/past mo",
    "Ever have 4/5 or more drinks every day",
]
df = df.drop(columns=binge_cols, errors="ignore")

In [20]:
print(df.shape)

(3480, 49)


## Change Smoked at least 100 cigarettes in life, Do you now smoke cigarettes? into smoking_status, and drop the original columns.

In [21]:
import numpy as np
import pandas as pd

smoked_100 = "Smoked at least 100 cigarettes in life"   # 1 = yes, 2 = no
smoke_now  = "Do you now smoke cigarettes?"             # 1 = every day, 2 = some days, 3 = not at all

# 1. Check the skip pattern: "Do you now smoke" is only asked if smoked 100+ cigarettes
print(pd.crosstab(df[smoked_100].fillna(-1), df[smoke_now].fillna(-1),
                  rownames=["Smoked 100+"], colnames=["Smoke now"]))

# 2. Smoking status
df["smoking_status"] = np.select(
    [df[smoked_100] == 2,                                   # never smoked 100 cigarettes
     (df[smoked_100] == 1) & (df[smoke_now] == 3),          # smoked before, not at all now
     (df[smoked_100] == 1) & df[smoke_now].isin([1, 2])],   # smokes every day or some days
    ["Never", "Former", "Current"],
    default=None,                                           # unknown
)

# 3. Store it as an ordered category (Never < Former < Current)
df["smoking_status"] = pd.Categorical(df["smoking_status"],
                                      categories=["Never", "Former", "Current"], ordered=True)

# 4. Check before dropping
print("\nSmoking status:")
print(df["smoking_status"].value_counts(dropna=False).sort_index())

# 5. Drop the 2 original columns
df = df.drop(columns=[smoked_100, smoke_now])
print(df.shape)

Smoke now    -1.0   1.0   2.0   3.0
Smoked 100+                        
-1.0            5     0     0     0
 1.0            0   280    96   929
 2.0         2170     0     0     0

Smoking status:
smoking_status
Never      2170
Former      929
Current     376
NaN           5
Name: count, dtype: int64
(3480, 48)


In [22]:
smoking_detail = ["# days smoked cigs during past 30 days",
                  "Avg # cigarettes/day during past 30 days",
                  "Cigarette Menthol indicator"]
df = df.drop(columns=smoking_detail, errors="ignore")

In [23]:
print(df.shape)

(3480, 45)


## Change Sleep hours - weekdays or workdays, Sleep hours - weekends into sleep_hr_avg, short_sleep and drop the original columns.

In [24]:
import pandas as pd

wkday = "Sleep hours - weekdays or workdays"
wkend = "Sleep hours - weekends"

# 1. Average nightly sleep: a week has 5 weekdays and 2 weekend days
df["sleep_hr_avg"] = (5 * df[wkday] + 2 * df[wkend]) / 7

# 2. Short sleep: less than 7 hours a night (1/0), missing stays missing
df["short_sleep"] = (df["sleep_hr_avg"] < 7).astype(float).where(df["sleep_hr_avg"].notna())

# 3. Check before dropping
print(df[[wkday, wkend, "sleep_hr_avg"]].describe().round(1))
print(f"% missing: {df['sleep_hr_avg'].isna().mean():.1%}")
print("\nShort sleep (< 7 hours):")
print(df["short_sleep"].value_counts(dropna=False))

# 4. Drop the 2 original columns
df = df.drop(columns=[wkday, wkend])
print(df.shape)

       Sleep hours - weekdays or workdays  Sleep hours - weekends  \
count                              3446.0                  3444.0   
mean                                  7.6                     8.2   
std                                   1.5                     1.6   
min                                   2.0                     2.0   
25%                                   7.0                     7.5   
50%                                   8.0                     8.0   
75%                                   8.5                     9.0   
max                                  14.0                    14.0   

       sleep_hr_avg  
count        3432.0  
mean            7.8  
std             1.4  
min             2.0  
25%             7.0  
50%             8.0  
75%             8.6  
max            14.0  
% missing: 1.4%

Short sleep (< 7 hours):
short_sleep
0.0    2708
1.0     724
NaN      48
Name: count, dtype: int64
(3480, 45)


In [25]:
clock_cols = ["Usual sleep time on weekdays or workdays", "Usual wake time on weekdays or workdays",
              "Usual sleep time on weekends", "Usual wake time on weekends"]
df = df.drop(columns=clock_cols, errors="ignore")

In [26]:
print(df.shape)

(3480, 41)


Ratio of family income to poverty and MEC exam weight contain values like 5.4e-79, change it to zero.

In [28]:
for col in ["Ratio of family income to poverty", "Full sample 2-year MEC exam weight"]:
    df.loc[df[col].abs() < 1e-10, col] = 0

## Add the response variable percentage change.

In [29]:
now = "Current self-reported weight (pounds)"
ago = "Self-reported weight - 1 yr ago (pounds)"
ht  = "Current self-reported height (inches)"

# Outcome: % weight change (negative = weight lost)
df["pct_wt_change"] = (df[now] - df[ago]) / df[ago] * 100

# Look at implausible changes before removing them
print(df.loc[df["pct_wt_change"].abs() > 50, [now, ago, "pct_wt_change"]].round(1))
df = df[df["pct_wt_change"].abs() <= 50]          # removes 6 people

# Weight-loss groups for descriptive tables
df["lost_5pct"] = (df["pct_wt_change"] <= -5).astype(int)

# Baseline BMI (before the change) from self-reported weight 1 year ago and height
df["bmi_1yr_ago"] = 703 * df[ago] / df[ht] ** 2

print(df["pct_wt_change"].describe().round(2))
print(df.shape)

      Current self-reported weight (pounds)  \
179                                   357.0   
227                                   230.0   
581                                   125.0   
1390                                  214.0   
2258                                  129.0   
3416                                  190.0   

      Self-reported weight - 1 yr ago (pounds)  pct_wt_change  
179                                      228.0           56.6  
227                                      138.0           66.7  
581                                      300.0          -58.3  
1390                                     469.0          -54.4  
2258                                     260.0          -50.4  
3416                                     120.0           58.3  
count    3474.00
mean       -1.39
std         9.12
min       -50.00
25%        -6.12
50%         0.00
75%         2.89
max        50.00
Name: pct_wt_change, dtype: float64
(3474, 44)


In [30]:
missing = pd.DataFrame({
    "n_missing":   df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
}).sort_values("pct_missing", ascending=False)

pd.set_option("display.max_rows", None)   # show every column
missing

,n_missing,pct_missing
Length of time in US,2761,79.5
Community/Government meals delivered,2059,59.3
Eat meals at Community/Senior center,2059,59.3
drinks_per_week,1026,29.5
depression_phq10,1013,29.2
phq9_severity,1013,29.2
phq9_score,1013,29.2
alcohol_status,1011,29.1
Upper Leg Length (cm),844,24.3
Upper Arm Length (cm),755,21.7


## Drop the columns with more than 30% missing data.

In [31]:
threshold = 30   # percent

pct_missing = df.isna().mean() * 100
cols_to_drop = pct_missing[pct_missing > threshold].index.tolist()

print(f"Columns with more than {threshold}% missing ({len(cols_to_drop)}):")
print(pct_missing[cols_to_drop].round(1).sort_values(ascending=False).to_string())

df = df.drop(columns=cols_to_drop)
print(f"\nNew shape: {df.shape}")

Columns with more than 30% missing (3):
Length of time in US                    79.5
Community/Government meals delivered    59.3
Eat meals at Community/Senior center    59.3

New shape: (3474, 41)


In [32]:
df.to_csv("nhanes_cleaned.csv", index=False)
print(f"Saved nhanes_cleaned.csv with shape {df.shape}")

Saved nhanes_cleaned.csv with shape (3474, 41)
